# Implementasi ETL Pipeline — Prediksi Ketersediaan Sepeda Citi Bike Jersey City
**Mata kuliah:** Big Data · **Tugas Minggu 10** · **Kelompok X**
Herpratama Rafi Putranto (101032300076) · Fandy Rifky Melani Saputra (101032300181)

Notebook ini menjalankan pipeline **Extract → Transform → Load** secara berurutan dengan kode yang sama dengan yang dijalankan otomatis di GitHub Actions
(repositori: https://github.com/rafiptrnto/citibike-jc-pipeline, paket `etl/`).

| Komponen | Isi |
|---|---|
| Sumber data | Trip data bulanan Citi Bike Jersey City (Amazon S3, CSV dalam ZIP), GBFS Citi Bike (API JSON, streaming), cuaca Open-Meteo (API JSON) |
| Target storage | PostgreSQL pada Supabase (proyek `citibike-etl`), tabel `dim_station`, `weather_hourly`, `fact_station_30min` |
| Bulan yang didemonstrasikan | Oktober 2026 (`202610`) untuk trip data dan cuaca, serta satu snapshot GBFS terbaru |

Urutan notebook: **1.** Persiapan · **2.** Extract · **3.** Transform · **4.** Load · **5.** Integrasi pipeline · **6.** Testing dan validasi · **7.** Ekspor dataset hasil transformasi.

## 1. Persiapan lingkungan dan koneksi ke target storage

In [ ]:
import os, sys, subprocess, getpass, datetime as dt

# Ambil kode pipeline dari repositori (modul etl/: config, extract, transform, load, quality, run_etl)
if os.path.basename(os.getcwd()) != "citibike-jc-pipeline":
    if not os.path.exists("citibike-jc-pipeline"):
        subprocess.run(["git", "clone", "-q", "https://github.com/rafiptrnto/citibike-jc-pipeline.git"], check=True)
    os.chdir("citibike-jc-pipeline")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "psycopg2-binary"], check=True)
sys.path.insert(0, os.getcwd())
print("Kode pipeline siap di:", os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
# Connection string Supabase (Session pooler) dimasukkan saat dijalankan, tidak disimpan di notebook.
os.environ["DATABASE_URL"] = getpass.getpass("Connection string Supabase: ")

import pandas as pd
from IPython.display import display
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
from etl import config, extract, transform as T, load, quality, run_etl

conn = load.connect()                       # gagal dengan pesan jelas jika koneksi ditolak
info_db = load.query_df(conn, "select current_database() as database, current_user as pengguna, "
                              "split_part(version(), ' ', 2) as versi_postgresql")
print("KONEKSI KE TARGET STORAGE BERHASIL")
display(info_db)
tabel = load.query_df(conn, """
    select 'dim_station' as tabel, count(*) as baris from public.dim_station
    union all select 'weather_hourly', count(*) from public.weather_hourly
    union all select 'fact_station_30min', count(*) from public.fact_station_30min""")
display(tabel)

## 2. Extract — pengambilan data dari sumber

Tiga sumber diekstraksi dengan `etl/extract.py` tanpa mengubah isinya:
1. **GBFS** (`extract_gbfs`): `station_information` dan `station_status` melalui API JSON (streaming).
2. **Trip data** (`extract_trip_month`): file ZIP bulanan dari bucket S3 `tripdata`, dibaca langsung menjadi DataFrame.
3. **Cuaca** (`extract_weather`): API Open-Meteo (arsip untuk tanggal lama, forecast untuk tanggal terbaru).

Fungsi pengambilan JSON memakai percobaan ulang (jeda 5, 15, 45 detik) untuk galat jaringan sementara.

In [ ]:
def inspeksi(df, nama, n=5):
    """Pemeriksaan awal: jumlah baris/kolom, tipe data, missing values, duplikat, contoh baris."""
    print(f"===== {nama} =====")
    print(f"Jumlah baris : {len(df):,}")
    print(f"Jumlah kolom : {df.shape[1]}")
    ringkas = pd.DataFrame({
        "tipe_data": df.dtypes.astype(str),
        "kosong": df.isna().sum(),
        "kosong_%": (df.isna().mean() * 100).round(2),
        "nilai_unik": df.astype(str).nunique()})
    display(ringkas)
    dup = int(df.astype(str).duplicated().sum())
    print(f"Baris duplikat penuh : {dup:,}")
    print(f"Total sel kosong     : {int(df.isna().sum().sum()):,}")
    display(df.head(n))
    return {"baris": len(df), "kolom": df.shape[1], "sel_kosong": int(df.isna().sum().sum()), "duplikat_penuh": dup}

# --- Sumber 1: GBFS (API JSON)
print("Proses Extract GBFS dimulai")
info_raw, status_raw, snapshot_utc = extract.extract_gbfs()
print(f"Data berhasil diekstraksi. Snapshot GBFS: {snapshot_utc}")
r_info = inspeksi(info_raw, "GBFS station_information")
r_status = inspeksi(status_raw, "GBFS station_status")

In [ ]:
# --- Sumber 2: trip data bulanan (CSV dalam ZIP di Amazon S3)
BULAN = "202610"
print(f"Proses Extract trip data {BULAN} dimulai")
trips_raw = extract.extract_trip_month(BULAN)
assert trips_raw is not None, "File trip bulan ini belum tersedia di S3"
print(f"Data berhasil diekstraksi dari {extract.nama_file_trip(BULAN)}")
r_trips = inspeksi(trips_raw, f"Trip data {BULAN}")

In [ ]:
# --- Sumber 3: cuaca per jam (API Open-Meteo)
awal_bln = pd.Timestamp(f"{BULAN[:4]}-{BULAN[4:]}-01")
akhir_bln = awal_bln + pd.offsets.MonthBegin(1)
print("Proses Extract cuaca dimulai")
cuaca_raw = extract.extract_weather(awal_bln.date(), (akhir_bln - pd.Timedelta(days=1)).date(),
                                    today=pd.Timestamp.now(tz=config.TZ).date())
print("Data berhasil diekstraksi")
r_cuaca = inspeksi(cuaca_raw, "Cuaca Open-Meteo")

In [ ]:
# Ringkasan kondisi awal data (pembanding untuk tahap Transform)
kondisi_awal = pd.DataFrame([
    ["GBFS station_information", "API JSON (GBFS 2.3)", r_info["baris"], r_info["kolom"], r_info["sel_kosong"], r_info["duplikat_penuh"]],
    ["GBFS station_status",      "API JSON (GBFS 2.3)", r_status["baris"], r_status["kolom"], r_status["sel_kosong"], r_status["duplikat_penuh"]],
    [f"Trip data {BULAN}",       "CSV dalam ZIP (S3)",  r_trips["baris"], r_trips["kolom"], r_trips["sel_kosong"], r_trips["duplikat_penuh"]],
    ["Cuaca Open-Meteo",         "API JSON",            r_cuaca["baris"], r_cuaca["kolom"], r_cuaca["sel_kosong"], r_cuaca["duplikat_penuh"]],
], columns=["Sumber", "Format", "Baris", "Kolom", "Sel kosong", "Duplikat penuh"])
display(kondisi_awal)

## 3. Transform — inspeksi, pembersihan, transformasi, validasi

### 3.1 Data inspection: permasalahan pada data hasil ekstraksi

In [ ]:
# Permasalahan pada trip data (sebelum transformasi)
t = trips_raw
mulai_t = pd.to_datetime(t["started_at"], errors="coerce", format="mixed")
akhir_t = pd.to_datetime(t["ended_at"], errors="coerce", format="mixed")
durasi = (akhir_t - mulai_t).dt.total_seconds()
sid_s = t["start_station_id"].astype("string").str.strip().str.upper()
sid_e = t["end_station_id"].astype("string").str.strip().str.upper()
masalah_trip = pd.DataFrame([
    ["started_at, ended_at", "Incorrect data type", f"Terbaca sebagai {t['started_at'].dtype} (teks), bukan datetime", "Tidak bisa dibulatkan ke slot 30 menit"],
    ["ride_id", "Duplicate records", f"{int(t['ride_id'].duplicated().sum()):,} baris ride_id berulang", "Perjalanan terhitung ganda"],
    ["started_at, ended_at", "Invalid values", f"{int((durasi < config.DURASI_MIN_DETIK).sum()):,} perjalanan < 1 menit dan {int((durasi > config.DURASI_MAX_DETIK).sum()):,} > 24 jam", "Mendistorsi jumlah perjalanan per slot"],
    ["started_at", "Invalid values", f"{int(((mulai_t < awal_bln) | (mulai_t >= akhir_bln)).sum()):,} perjalanan dimulai di luar bulan file", "Bukan bagian dari bulan yang dimuat"],
    ["start_station_id", "Missing values", f"{int(t['start_station_id'].isna().sum()):,} kosong ({t['start_station_id'].isna().mean()*100:.2f}%)", "Tidak bisa dipetakan ke stasiun"],
    ["end_station_id", "Missing values", f"{int(t['end_station_id'].isna().sum()):,} kosong ({t['end_station_id'].isna().mean()*100:.2f}%)", "Tidak bisa dipetakan ke stasiun"],
    ["start/end_station_id", "Inconsistent data", f"{int((~sid_s.fillna('').str.match(r'^(JC|HB)')).sum()):,} ID awal bukan JC/HB (mis. stasiun New York)", "Di luar cakupan studi kasus"],
    ["rideable_type, member_casual, ride_id", "Irrelevant attributes", "Tidak diperlukan pada tabel fakta per stasiun per 30 menit", "Dibuang saat agregasi"],
], columns=["Atribut", "Jenis masalah", "Temuan", "Dampak"])
display(masalah_trip)

In [ ]:
# Permasalahan pada GBFS (sebelum transformasi)
s = info_raw
sn = s["short_name"].astype("string").str.strip().str.upper()
jc_hb = sn.str.match(r"^(JC|HB)")
masalah_gbfs = pd.DataFrame([
    ["short_name", "Inconsistent data", f"{int((~jc_hb.fillna(False)).sum()):,} dari {len(s):,} stasiun bukan JC/HB (New York)", "Dibuang (aturan R1)"],
    ["short_name", "Inconsistent data", f"{int(sn[jc_hb.fillna(False)].duplicated(keep=False).sum()):,} stasiun JC/HB dengan short_name ganda", "Pemetaan ambigu, dibuang"],
    ["capacity", "Invalid values", f"{int((s['capacity'].fillna(0) <= 0).sum()):,} stasiun berkapasitas 0 / kosong", "Stasiun tidak operasional, dibuang"],
    ["station_id (UUID)", "Inconsistent data", "ID live (UUID) berbeda dengan ID trip (JC032, HB612)", "Dipetakan lewat short_name (aturan R2)"],
    ["num_*_available", "Perlu agregasi", "Snapshot setiap beberapa menit, bukan per 30 menit", "Diagregasi per slot 30 menit (R3, R6)"],
    ["is_installed", "Invalid values", f"{int((status_raw['is_installed'] != 1).sum()):,} stasiun tidak terpasang", "Dibuang"],
], columns=["Atribut", "Jenis masalah", "Temuan", "Dampak"])
display(masalah_gbfs)
print(f"Cuaca: duplikat jam = {int(cuaca_raw.duplicated('hour_ts').sum()):,}; nilai kosong = {int(cuaca_raw[['temperature_2m','precipitation','wind_speed_10m']].isna().sum().sum()):,}")

### 3.2 Data cleaning dan transformation
Aturan yang diterapkan (`etl/transform.py`):

| Kode | Aturan | Alasan |
|---|---|---|
| R1 | Saring stasiun Jersey City dan Hoboken yang aktif | Studi kasus hanya Jersey City |
| R2 | Petakan ID live (short_name) ke ID historical; cadangan lewat nama | Kunci stasiun harus sama pada kedua sumber |
| R3 | Konversi waktu ke America/New_York dan bulatkan ke slot 30 menit | Zona waktu data trip dan analisis |
| R4 | Buang duplikat (`ride_id`; `snapshot_time` + `station_id`) | Hindari perhitungan ganda |
| R5 | Buang perjalanan berdurasi < 1 menit atau > 24 jam | Nilai tidak valid |
| R6 | Agregasi per stasiun per slot (grid lengkap untuk trip) | Struktur tabel fakta |
| R7 | Gabungkan cuaca per jam | Fitur prediksi |

In [ ]:
print("Proses Transform trip data dimulai")
dim = load.baca_dim(conn)                                      # dimensi stasiun dari target (kunci pemetaan)
trips_bersih, ringkas = T.bersihkan_trips(trips_raw, BULAN)    # R3, R4, R5 + konversi tipe data
trips_map, dim_baru = T.petakan_stasiun_trip(trips_bersih, dim)  # R1, R2
fakta_trip = T.agregasi_trips(trips_map, BULAN)                # R3, R6
cuaca = T.transform_weather(cuaca_raw)                         # R7 (cuaca per jam)
fakta_trip = T.tambah_kalender(T.gabung_cuaca(fakta_trip, cuaca))  # R7 + fitur kalender
print("Data berhasil dibersihkan dan ditransformasi")
print("Ringkasan pembersihan:")
display(pd.Series(ringkas, name="jumlah").to_frame())
print(f"Stasiun baru (hanya ada di historical) yang ditambahkan ke dim_station: {len(dim_baru)}")
display(fakta_trip.head())
display(fakta_trip.dtypes.astype(str).to_frame("tipe_data").T)

In [ ]:
# Transformasi snapshot GBFS: R1, R2, R3, R4, R6
dim_live = T.transform_station_info(info_raw)
snaps = T.transform_status_snapshot(status_raw, dim_live)
agg = T.aggregate_snapshots(snaps)
fakta_stream = T.gabung_status(agg)
print(f"GBFS: {len(status_raw):,} stasiun mentah -> {len(dim_live)} stasiun JC/HB aktif -> {len(fakta_stream)} baris slot 30 menit")
display(fakta_stream.head())

### 3.3 Data validation

In [ ]:
validasi = []
def cek(nama, ok, aktual):
    validasi.append({"Pemeriksaan": nama, "Hasil aktual": aktual, "Status": "Berhasil" if ok else "Gagal"})

n_slot = len(pd.date_range(awal_bln, akhir_bln, freq="30min", inclusive="left"))
n_st = fakta_trip["station_id"].nunique()
kolom_inti = ["station_id", "waktu", "tanggal", "jam", "hari", "akhir_pekan", "suhu_c", "hujan_mm", "angin_kmh",
              "trips_keluar", "trips_masuk", "net_flow_trip"]
cek("Jumlah baris = stasiun x slot 30 menit", len(fakta_trip) == n_st * n_slot, f"{len(fakta_trip):,} = {n_st} x {n_slot:,}")
cek("Tidak ada duplikat (station_id, waktu)", not fakta_trip.duplicated(["station_id", "waktu"]).any(), f"{int(fakta_trip.duplicated(['station_id','waktu']).sum())} duplikat")
cek("Total trips_keluar = trip bersih berstasiun asal valid", fakta_trip["trips_keluar"].sum() == trips_map["start_sid"].notna().sum(), f"{int(fakta_trip['trips_keluar'].sum()):,} vs {int(trips_map['start_sid'].notna().sum()):,}")
cek("Total trips_masuk = trip bersih berstasiun tujuan valid", fakta_trip["trips_masuk"].sum() == trips_map["end_sid"].notna().sum(), f"{int(fakta_trip['trips_masuk'].sum()):,} vs {int(trips_map['end_sid'].notna().sum()):,}")
cek("net_flow_trip = trips_masuk - trips_keluar", bool((fakta_trip["net_flow_trip"] == fakta_trip["trips_masuk"] - fakta_trip["trips_keluar"]).all()), "konsisten pada semua baris")
cek("Semua waktu tepat di awal slot 30 menit", bool(fakta_trip["waktu"].dt.minute.isin([0, 30]).all() and (fakta_trip["waktu"].dt.second == 0).all()), "menit 0/30, detik 0")
cek("Tidak ada nilai negatif pada jumlah perjalanan", bool((fakta_trip[["trips_keluar", "trips_masuk"]] >= 0).all().all()), "min = " + str(int(fakta_trip[["trips_keluar", "trips_masuk"]].min().min())))
cek("Kolom inti tidak memiliki nilai kosong", int(fakta_trip[kolom_inti].isna().sum().sum()) == 0, f"{int(fakta_trip[kolom_inti].isna().sum().sum())} sel kosong")
cek("Tipe data waktu = datetime, jumlah = bilangan bulat", str(fakta_trip["waktu"].dtype).startswith("datetime64") and fakta_trip["trips_keluar"].dtype.kind == "i", f"waktu: {fakta_trip['waktu'].dtype}, trips_keluar: {fakta_trip['trips_keluar'].dtype}")
cek("Stasiun hanya Jersey City / Hoboken", bool(fakta_trip["station_id"].str.match(r"^(JC|HB)").all()), f"{n_st} stasiun")
cek("GBFS: kosong/penuh konsisten dengan jumlah sepeda/dok", bool(((fakta_stream["kosong"] == (fakta_stream["sepeda_tersedia"] == 0)) & (fakta_stream["penuh"] == (fakta_stream["dok_tersedia"] == 0))).all()), f"{len(fakta_stream)} baris diperiksa")
hasil_validasi = pd.DataFrame(validasi)
display(hasil_validasi)
assert (hasil_validasi["Status"] == "Berhasil").all(), "Ada pemeriksaan validasi yang gagal"

### 3.4 Perbandingan kondisi data sebelum dan sesudah transformasi

In [ ]:
sebelum_sel_kosong = int(trips_raw.isna().sum().sum())
setelah_sel_kosong = int(fakta_trip[kolom_inti].isna().sum().sum())
bandingkan = pd.DataFrame([
    ["Jumlah baris", f"{len(trips_raw):,} perjalanan", f"{len(trips_bersih):,} perjalanan bersih, lalu {len(fakta_trip):,} baris agregat", "Pembersihan lalu agregasi ke stasiun x slot 30 menit"],
    ["Jumlah kolom", trips_raw.shape[1], fakta_trip.shape[1], "Atribut perjalanan diganti atribut agregat, kalender, dan cuaca"],
    ["Sel kosong", f"{sebelum_sel_kosong:,}", f"{setelah_sel_kosong:,} (kolom inti)", "Perjalanan tanpa stasiun tidak masuk agregat stasiun"],
    ["Duplikat ride_id", f"{ringkas['duplikat_ride_id']:,}", "0", "Duplikat dibuang (R4)"],
    ["Durasi tidak wajar", f"{ringkas['durasi_tidak_wajar']:,}", "0", "Dibuang (R5)"],
    ["Di luar bulan file", f"{ringkas['di_luar_bulan']:,}", "0", "Dibuang"],
    ["Tipe started_at / ended_at", "object (teks)", "datetime64 -> waktu (awal slot)", "Konversi tipe data (R3)"],
], columns=["Aspek", "Sebelum", "Sesudah", "Keterangan"])
display(bandingkan)
selisih = len(trips_raw) - len(trips_bersih)
print(f"Selisih {selisih:,} baris trip = waktu tidak valid {ringkas['waktu_tidak_valid']:,} + duplikat {ringkas['duplikat_ride_id']:,} "
      f"+ durasi tidak wajar {ringkas['durasi_tidak_wajar']:,} + di luar bulan {ringkas['di_luar_bulan']:,}")
assert selisih == ringkas["waktu_tidak_valid"] + ringkas["duplikat_ride_id"] + ringkas["durasi_tidak_wajar"] + ringkas["di_luar_bulan"]

## 4. Load — penyimpanan ke target storage (Supabase PostgreSQL)

**Mekanisme: upsert** (`INSERT ... ON CONFLICT (kunci) DO UPDATE`). Pipeline berjalan berulang (setiap 10 menit untuk streaming, harian untuk batch), sehingga
baris yang sama tidak boleh tertulis ganda: kunci `(station_id, waktu)` pada tabel fakta, `station_id` pada `dim_station`, dan `hour_ts` pada `weather_hourly`.
Kolom dari trip data dan kolom dari streaming berada pada baris yang sama tetapi diperbarui oleh job yang berbeda.

In [ ]:
# Struktur penyimpanan yang digunakan (dibaca dari katalog database)
struktur = load.query_df(conn, """
    select table_name as tabel, ordinal_position as no, column_name as kolom, data_type as tipe,
           is_nullable as boleh_kosong
    from information_schema.columns
    where table_schema = 'public' and table_name in ('dim_station','weather_hourly','fact_station_30min')
    order by table_name, ordinal_position""")
display(struktur.groupby("tabel").size().to_frame("jumlah_kolom"))
display(struktur[struktur["tabel"] == "fact_station_30min"])
kunci = load.query_df(conn, """
    select tc.table_name as tabel, tc.constraint_type as jenis, string_agg(k.column_name, ', ' order by k.ordinal_position) as kolom
    from information_schema.table_constraints tc join information_schema.key_column_usage k using (constraint_name, table_schema)
    where tc.table_schema = 'public' and tc.table_name in ('dim_station','weather_hourly','fact_station_30min')
      and tc.constraint_type in ('PRIMARY KEY','FOREIGN KEY') group by 1, 2 order by 1, 2""")
display(kunci)

In [ ]:
print("Proses Load dimulai")
n_dim = load.load_dim_historical(conn, dim_baru)
n_cuaca = load.load_weather(conn, cuaca)
n_fakta = load.load_fact_trips(conn, fakta_trip)
load.catat_batch(conn, extract.nama_file_trip(BULAN), BULAN, ringkas, n_fakta)
conn.commit()
print(f"Data berhasil disimpan ke target storage: dim_station +{n_dim} baru, weather_hourly {n_cuaca:,} jam, fact_station_30min {n_fakta:,} baris")

### 4.1 Pemeriksaan hasil penyimpanan dan konsistensi data

In [ ]:
# Baca kembali dari target storage dan bandingkan dengan hasil transformasi
db = load.query_df(conn, """
    select station_id, waktu, trips_keluar, trips_masuk, net_flow_trip, suhu_c, hujan_mm, angin_kmh, tanggal, jam, hari, akhir_pekan
    from public.fact_station_30min
    where waktu >= %s and waktu < %s and trips_keluar is not null""", (awal_bln.to_pydatetime(), akhir_bln.to_pydatetime()))
db["waktu"] = pd.to_datetime(db["waktu"])
for c in ["trips_keluar", "trips_masuk", "net_flow_trip", "suhu_c", "hujan_mm", "angin_kmh"]:
    db[c] = pd.to_numeric(db[c])

import numpy as np
m = fakta_trip.merge(db, on=["station_id", "waktu"], how="outer", suffixes=("_etl", "_db"), indicator=True)
sama_kunci = (m["_merge"] == "both").sum()
beda = {}
for c in ["trips_keluar", "trips_masuk", "net_flow_trip"]:
    beda[c] = int((m[f"{c}_etl"] != m[f"{c}_db"]).sum())
for c in ["suhu_c", "hujan_mm", "angin_kmh"]:
    beda[c] = int((~np.isclose(m[f"{c}_etl"].astype(float), m[f"{c}_db"].astype(float), atol=1e-3, equal_nan=True)).sum())
konsistensi = pd.DataFrame([
    ["Baris hasil transformasi", f"{len(fakta_trip):,}"],
    ["Baris terbaca kembali dari target storage", f"{len(db):,}"],
    ["Baris dengan kunci cocok di kedua sisi", f"{int(sama_kunci):,}"],
    ["Baris hanya ada di salah satu sisi", f"{int((m['_merge'] != 'both').sum()):,}"],
] + [[f"Selisih nilai kolom {c}", f"{v:,}"] for c, v in beda.items()], columns=["Pemeriksaan", "Hasil"])
display(konsistensi)
assert len(db) == len(fakta_trip) and all(v == 0 for v in beda.values()) and (m["_merge"] == "both").all()
print("Data di target storage identik dengan hasil transformasi.")

In [ ]:
# Contoh data yang tersimpan di target storage (dibaca dengan SQL)
display(load.query_df(conn, """
    select station_id, waktu, trips_keluar, trips_masuk, net_flow_trip, sepeda_tersedia, dok_tersedia, suhu_c, sumber
    from public.fact_station_30min
    where station_id = 'JC009' and waktu between '2026-10-24 07:30' and '2026-10-24 10:00' order by waktu"""))
display(load.query_df(conn, "select sumber, count(*) as baris, min(waktu) as dari, max(waktu) as sampai from public.fact_station_30min group by sumber order by 1"))

## 5. Integrasi ETL Pipeline

Tiga tahap digabung menjadi satu fungsi `etl_trips(conn, bulan)` yang berjalan berurutan: **Extract → Transform → Load**.
Output setiap tahap menjadi input tahap berikutnya; Transform hanya berjalan jika Extract berhasil, dan Load hanya jika Transform berhasil
(pengecualian Python menghentikan alur dan tahap yang gagal dicatat pada log). Setiap run dicatat ke tabel `etl_log_run`.

```
Sumber (S3 trip ZIP, Open-Meteo, GBFS)
   -> EXTRACT  (extract.py)   : unduh, baca ke DataFrame, retry jaringan
   -> TRANSFORM(transform.py) : inspeksi, R1-R7, validasi
   -> LOAD     (load.py)      : upsert ke Supabase PostgreSQL, catat log
   -> Target storage: dim_station, weather_hourly, fact_station_30min, etl_log_run
```

In [ ]:
def log(pesan):
    print(f"[{dt.datetime.now():%H:%M:%S}] {pesan}")

def etl_trips(conn, bulan):
    """Pipeline ETL trip data satu bulan: Extract -> Transform -> Load (+ logging)."""
    mulai, tahap = load.sekarang(), "Extract"
    try:
        # ---------------- EXTRACT
        log(f"Proses Extract dimulai ({bulan})")
        raw = extract.extract_trip_month(bulan)
        if raw is None:
            raise RuntimeError("file trip belum tersedia di S3")
        awal = pd.Timestamp(f"{bulan[:4]}-{bulan[4:]}-01"); akhir = awal + pd.offsets.MonthBegin(1)
        cuaca_mentah = extract.extract_weather(awal.date(), (akhir - pd.Timedelta(days=1)).date(),
                                               today=pd.Timestamp.now(tz=config.TZ).date())
        log(f"Data berhasil diekstraksi: {len(raw):,} perjalanan, {len(cuaca_mentah):,} baris cuaca")
        # ---------------- TRANSFORM
        tahap = "Transform"
        log("Proses Transform dimulai")
        dim_ = load.baca_dim(conn)
        bersih, ring = T.bersihkan_trips(raw, bulan)
        bersih, dim_baru_ = T.petakan_stasiun_trip(bersih, dim_)
        fakta = T.agregasi_trips(bersih, bulan)
        w = T.transform_weather(cuaca_mentah)
        fakta = T.tambah_kalender(T.gabung_cuaca(fakta, w))
        assert not fakta.duplicated(["station_id", "waktu"]).any(), "kunci ganda pada hasil transformasi"
        log(f"Data berhasil dibersihkan dan ditransformasi: {len(bersih):,} perjalanan bersih -> {len(fakta):,} baris agregat")
        # ---------------- LOAD
        tahap = "Load"
        log("Proses Load dimulai")
        load.load_dim_historical(conn, dim_baru_)
        load.load_weather(conn, w)
        n = load.load_fact_trips(conn, fakta)
        load.catat_batch(conn, extract.nama_file_trip(bulan), bulan, ring, n)
        load.catat_log(conn, "notebook_trips", mulai, "sukses", len(raw), len(fakta), n, f"notebook ETL_Pipeline, bulan {bulan}")
        conn.commit()
        log(f"Data berhasil disimpan ke target storage: {n:,} baris")
        log("ETL Pipeline selesai dijalankan")
        return {"extract": len(raw), "bersih": len(bersih), "transform": len(fakta), "load": n}
    except Exception as e:
        conn.rollback()
        log(f"ETL Pipeline GAGAL pada tahap {tahap}: {type(e).__name__}")
        load.catat_log(conn, "notebook_trips", mulai, "gagal", pesan=f"tahap {tahap}: {type(e).__name__}")
        conn.commit()
        raise

hitung_awal = int(load.query_df(conn, "select count(*) n from public.fact_station_30min")["n"].iloc[0])
run1 = etl_trips(conn, BULAN)
print(run1)

In [ ]:
# Pipeline streaming (GBFS) memakai job yang sama dengan GitHub Actions: Extract -> Transform -> Load dalam satu fungsi
print("Menjalankan job streaming (satu snapshot GBFS terbaru)")
hasil_stream = run_etl._jalankan(conn, "stream", run_etl.job_stream)
display(load.query_df(conn, """
    select id, job, mulai, status, baris_extract as extract, baris_transform as transform, baris_load as load, left(pesan, 80) as pesan
    from public.etl_log_run order by id desc limit 6"""))

## 6. Testing dan validasi

In [ ]:
# Uji unit untuk setiap aturan transformasi (tanpa internet dan database)
hasil_pytest = subprocess.run([sys.executable, "-m", "pytest", "-v", "--no-header", "-p", "no:cacheprovider"],
                              capture_output=True, text=True)
print(hasil_pytest.stdout[-3500:])

In [ ]:
# Pengujian per tahap dan end-to-end
uji = []
def uji_tambah(tahap, aspek, harapan, aktual, ok):
    uji.append({"Tahapan": tahap, "Aspek yang diuji": aspek, "Hasil yang diharapkan": harapan, "Hasil aktual": aktual,
                "Status": "Berhasil" if ok else "Gagal"})

wajib_trip = {"ride_id", "started_at", "ended_at", "start_station_id", "end_station_id", "start_lat", "start_lng", "end_lat", "end_lng"}
batch = load.query_df(conn, "select baris_mentah, baris_bersih, baris_load from public.etl_batch_loaded where bulan = %s", (BULAN,)).iloc[0]
# EXTRACT
uji_tambah("Extract", "Akses sumber data", "Ketiga sumber dapat diakses", "GBFS, S3, Open-Meteo berhasil", len(status_raw) > 0 and len(trips_raw) > 0 and len(cuaca_raw) > 0)
uji_tambah("Extract", "Pembacaan data", "Data berhasil diambil", f"{len(trips_raw):,} trip, {len(info_raw):,} stasiun GBFS, {len(cuaca_raw):,} jam cuaca", len(trips_raw) > 0)
uji_tambah("Extract", "Jumlah baris sesuai sumber", "Sama dengan hasil ekstraksi yang tercatat pipeline", f"{len(trips_raw):,} = {int(batch['baris_mentah']):,}", len(trips_raw) == int(batch["baris_mentah"]))
uji_tambah("Extract", "Atribut yang dibutuhkan", "Kolom wajib tersedia", f"{len(wajib_trip & set(trips_raw.columns))} dari {len(wajib_trip)} kolom wajib", wajib_trip <= set(trips_raw.columns))
# TRANSFORM
uji_tambah("Transform", "Penanganan duplikasi", "Tidak ada ride_id ganda setelah cleaning", f"{int(trips_bersih['ride_id'].duplicated().sum())} duplikat (sebelumnya {ringkas['duplikat_ride_id']:,})", not trips_bersih["ride_id"].duplicated().any())
dur_ok = ((trips_bersih["ended_at"] - trips_bersih["started_at"]).dt.total_seconds().between(config.DURASI_MIN_DETIK, config.DURASI_MAX_DETIK)).all()
uji_tambah("Transform", "Penanganan nilai tidak valid", "Semua durasi 1 menit - 24 jam", "semua durasi dalam rentang", bool(dur_ok))
uji_tambah("Transform", "Penanganan missing values", "Kolom inti agregat tanpa nilai kosong", f"{setelah_sel_kosong} sel kosong", setelah_sel_kosong == 0)
uji_tambah("Transform", "Tipe data dan struktur", "waktu datetime, jumlah integer", f"{fakta_trip['waktu'].dtype}, {fakta_trip['trips_keluar'].dtype}", str(fakta_trip["waktu"].dtype).startswith("datetime64"))
uji_tambah("Transform", "Perubahan jumlah baris dijelaskan", "Selisih = baris yang dibuang oleh aturan", f"{selisih:,} = {ringkas['waktu_tidak_valid']} + {ringkas['duplikat_ride_id']} + {ringkas['durasi_tidak_wajar']} + {ringkas['di_luar_bulan']}", selisih == ringkas["waktu_tidak_valid"] + ringkas["duplikat_ride_id"] + ringkas["durasi_tidak_wajar"] + ringkas["di_luar_bulan"])
uji_tambah("Transform", "Validasi hasil transformasi", "Seluruh pemeriksaan validasi lolos", f"{int((hasil_validasi['Status']=='Berhasil').sum())}/{len(hasil_validasi)} lolos", bool((hasil_validasi["Status"] == "Berhasil").all()))
# LOAD
n_db = int(load.query_df(conn, "select count(*) n from public.fact_station_30min where waktu >= %s and waktu < %s and trips_keluar is not null", (awal_bln.to_pydatetime(), akhir_bln.to_pydatetime()))["n"].iloc[0])
uji_tambah("Load", "Koneksi ke target storage", "Program terhubung ke Supabase", "terhubung (lihat bagian 1)", conn.closed == 0)
uji_tambah("Load", "Penyimpanan data", "Data berhasil disimpan tanpa galat", f"{n_fakta:,} baris di-upsert", n_fakta == len(fakta_trip))
uji_tambah("Load", "Jumlah data tersimpan", "Sama dengan hasil transformasi", f"{n_db:,} = {len(fakta_trip):,}", n_db == len(fakta_trip))
uji_tambah("Load", "Pembacaan kembali", "Data dapat dibaca dan nilainya identik", "0 selisih nilai, 0 baris hilang", len(db) == len(fakta_trip) and all(v == 0 for v in beda.values()))
# END-TO-END: jalankan sekali lagi, hasil harus identik (idempotent) tanpa baris ganda
run2 = etl_trips(conn, BULAN)
n_db2 = int(load.query_df(conn, "select count(*) n from public.fact_station_30min where waktu >= %s and waktu < %s and trips_keluar is not null", (awal_bln.to_pydatetime(), akhir_bln.to_pydatetime()))["n"].iloc[0])
log_terakhir = load.query_df(conn, "select status from public.etl_log_run where job = 'notebook_trips' order by id desc limit 1")["status"].iloc[0]
uji_tambah("End-to-End", "Eksekusi pipeline", "Seluruh tahapan berjalan berurutan tanpa galat", f"status log: {log_terakhir}", log_terakhir == "sukses")
uji_tambah("End-to-End", "Run ulang (idempotent)", "Jumlah baris tidak bertambah", f"{n_db2:,} = {n_db:,}", n_db2 == n_db and run2 == run1)
uji_tambah("End-to-End", "Data akhir pada target storage", "Sesuai kebutuhan studi kasus", "108+ stasiun x 1.488 slot Oktober tersimpan", n_db2 == len(fakta_trip))
tabel_uji = pd.DataFrame(uji)
display(tabel_uji)
print("Ringkasan:", tabel_uji["Status"].value_counts().to_dict())

jumlah_tahap = pd.DataFrame([
    ["Setelah Extract", f"{len(trips_raw):,}", "Perjalanan mentah pada file trip data 202610"],
    ["Setelah Transform", f"{len(fakta_trip):,}", f"{len(trips_bersih):,} perjalanan bersih diagregasi menjadi {n_st} stasiun x {n_slot:,} slot"],
    ["Setelah Load", f"{n_db2:,}", "Baris fakta yang terbaca kembali dari Supabase"],
], columns=["Tahapan", "Jumlah baris", "Keterangan"])
display(jumlah_tahap)

In [ ]:
# Uji kualitas data pada target storage (11 pemeriksaan, hasil tersimpan di etl_quality_check)
hasil_q, gagal_q = quality.jalankan(conn)
display(pd.DataFrame(hasil_q, columns=["pemeriksaan", "lolos", "nilai", "kritis"]))
print(f"{sum(1 for h in hasil_q if h[1])}/{len(hasil_q)} pemeriksaan lolos; kegagalan kritis: {gagal_q or 'tidak ada'}")

## 7. Dataset hasil transformasi

In [ ]:
# Ekspor seluruh tabel fakta (hasil transformasi yang tersimpan) ke CSV
with open("transformed_dataset.csv", "w") as f, conn.cursor() as cur:
    cur.copy_expert("COPY (select * from public.fact_station_30min order by station_id, waktu) TO STDOUT WITH CSV HEADER", f)
ds = pd.read_csv("transformed_dataset.csv", nrows=5)
n_csv = sum(1 for _ in open("transformed_dataset.csv")) - 1
print(f"transformed_dataset.csv: {n_csv:,} baris, {ds.shape[1]} kolom, {os.path.getsize('transformed_dataset.csv')/1e6:.1f} MB")
display(ds)

In [ ]:
# (Opsional) simpan ke Google Drive agar mudah diunduh
from google.colab import drive
drive.mount("/content/drive")
tujuan = "/content/drive/MyDrive/MATPIL BIG DATA/tugas_minggu10"
os.makedirs(tujuan, exist_ok=True)
subprocess.run(["cp", "transformed_dataset.csv", tujuan], check=True)
subprocess.run(["gzip", "-kf", "transformed_dataset.csv"], check=True)
subprocess.run(["cp", "transformed_dataset.csv.gz", tujuan], check=True)
print(os.listdir(tujuan))

In [ ]:
conn.close()
print('Koneksi ditutup. Pipeline selesai.')